# PP-2: Identify precinct neighbors
- Identify two precincts as neighbors if they share a common boundary of at least 200 feet and the edges of each precinct are within 200 feet of its neighbors’ edges. If possible, try to locate a data source for which this computation is already done.

In [58]:
import pandas as pd
import geopandas as gpd
import maup

## GA

In [59]:
ga_df = gpd.read_file("output/Georgia/seawulf_district_maup.gpkg")
ga_df.head()

,UNIQUE_ID,COUNTYFP,Kamala D. Harris,Donald J. Trump,Other_candidates,Total_votes,White_population,Black_population,Latino_population,Other_population,Total_population,District,geometry
0,APPLING-:-1B,001,108,921,2,1031,1074.216587,102.982480,55.003760,3.618320,1235.821146,1,"MULTIPOLYGON (((375680.672 3535246.721, 375706..."
1,APPLING-:-1C,001,67,724,0,791,1191.912652,472.673691,42.287709,28.713142,1735.587193,1,"MULTIPOLYGON (((378693.089 3524550.847, 378692..."
2,APPLING-:-2,001,782,541,4,1327,1235.612661,788.224241,96.849558,32.601383,2153.287844,1,"MULTIPOLYGON (((380005.78 3525000.183, 380113...."
3,APPLING-:-3A1,001,31,617,0,648,705.543283,121.434717,146.434230,7.825296,981.237527,1,"MULTIPOLYGON (((377772.229 3534595.104, 377753..."
4,APPLING-:-3C,001,246,934,3,1183,1427.934438,416.186299,57.464584,60.117966,1961.703287,1,"MULTIPOLYGON (((386430.886 3520109.322, 386469..."


In [60]:
columns = ['UNIQUE_ID', 'COUNTYFP', 'geometry']
ga_df = ga_df[columns]

In [61]:
print(ga_df.crs)
ga_df = ga_df.to_crs(epsg=2239)
print(ga_df.crs)

EPSG:32617
EPSG:2239


In [62]:
ga_df = ga_df.reset_index(drop=True)

In [64]:
sindex = ga_df.sindex
neighbors = []
type(sindex)

geopandas.sindex.SpatialIndex

In [65]:
for i, precinct in ga_df.iterrows():
    # Find candidates whose bounding boxes intersect (or are within 200ft)
    candidates = list(sindex.query(precinct.geometry, predicate="intersects"))
    curr_geo = ga_df['geometry'].iloc[i]
    # print(candidates)
    for j in candidates:
        if j <= i:
            continue
        
        other = ga_df.iloc[j]
        other_geo = ga_df['geometry'].iloc[j]

        # Compute shared boundary
        shared = precinct.geometry.boundary.intersection(other.geometry.boundary)
        shared_length = shared.length  # in feet if CRS is in feet
        
        if shared_length >= 200 and curr_geo.distance(other_geo) <= 200:
            neighbors.append({
                "precinct_a": precinct["UNIQUE_ID"],
                "precinct_b": other["UNIQUE_ID"],
                "shared_boundary_ft": shared_length
            })
        


In [70]:
candidates

[np.int64(2565),
 np.int64(2564),
 np.int64(2710),
 np.int64(726),
 np.int64(729),
 np.int64(2076),
 np.int64(980),
 np.int64(2714),
 np.int64(2719),
 np.int64(2071)]

In [8]:
print(f"total neighbors: {len(neighbors)}")

total neighbors: 7687


In [19]:
edges = pd.DataFrame(neighbors)
edges.head()

,precinct_a,precinct_b,shared_boundary_ft
0,APPLING-:-1B,APPLING-:-2,5224.186963
1,APPLING-:-1B,APPLING-:-1C,66404.865904
2,APPLING-:-1B,APPLING-:-3A1,50194.563951
3,APPLING-:-1B,JEFF DAVIS-:-ALTAMAHA 2,34852.129288
4,APPLING-:-1B,TOOMBS-:-43 CEDAR CROSSING,69146.164428


In [20]:
a_counts = edges["precinct_a"].value_counts()
b_counts = edges["precinct_b"].value_counts()

neighbor_counts = a_counts.add(b_counts, fill_value=0)

In [21]:
neighbor_counts = neighbor_counts.reset_index()
neighbor_counts.columns = ["UNIQUE_ID", "num_neighbors"]

In [26]:
neighbor_counts.to_csv("output/Georgia/ga_precinct_neighbors_cnt.csv", index=False)

In [27]:
edges.to_csv("output/Georgia/ga_precinct_neighbors_map.csv", index=False)

## AR

In [30]:
ar_df = gpd.read_file("output/Arkansas/ar_seawulf_maup.gpkg")
ar_df.head()

,Unique_ID,COUNTYFP,Kamala D. Harris,Donald J. Trump,Other_candidates,Total_votes,White_population,Black_population,Latino_population,Other_population,Total_population,District,geometry
0,05001-81 - Stuttgart 2,001,293,639,19,951,486.169395,356.604974,32.728840,0.738939,876.242147,1,"MULTIPOLYGON (((2195716.192 30309.368, 2195879..."
1,05001-36 - Gillett Ward 3,001,25,39,2,66,63.524419,10.565301,0.982819,2.213607,77.286145,1,"POLYGON ((2215708.638 -7028.264, 2215708.777 -..."
2,05001-53 - Dewitt 2,001,49,188,5,242,230.348940,133.055168,0.119868,4.781286,368.305262,1,"POLYGON ((2217343.007 10978.738, 2217328.353 1..."
3,05001-51 - DeWitt 1,001,101,61,5,167,332.713262,201.772191,0.000000,7.098115,541.583568,1,"POLYGON ((2218354.249 12107.388, 2218342.578 1..."
4,05001-55 - Dewitt 3,001,42,272,3,317,36.167855,9.010094,0.713321,0.281977,46.173247,1,"POLYGON ((2216959.397 14106.194, 2216978.998 1..."


In [32]:
columns = ['Unique_ID', 'COUNTYFP', 'geometry']
ar_df = ar_df[columns]

In [33]:
print(ar_df.crs)
ar_df = ar_df.to_crs(epsg=6411)
print(ar_df.crs)

EPSG:26954
EPSG:6411


In [36]:
ar_df = ar_df.reset_index(drop=True)

In [42]:
sindex = ar_df.sindex
ar_neighbors = []

In [43]:
for i, precinct in ar_df.iterrows():
    # Find candidates whose bounding boxes intersect (or are within 200ft)
    candidates = list(sindex.query(precinct.geometry, predicate="intersects"))
    curr_geo = ar_df['geometry'].iloc[i]
    # print(candidates)
    for j in candidates:
        if j <= i:
            continue
        
        other = ar_df.iloc[j]
        other_geo = ar_df['geometry'].iloc[j]

        # Compute shared boundary
        shared = precinct.geometry.boundary.intersection(other.geometry.boundary)
        shared_length = shared.length  # in feet if CRS is in feet
        
        if shared_length >= 200 and curr_geo.distance(other_geo) <= 200:
            ar_neighbors.append({
                "precinct_a": precinct["Unique_ID"],
                "precinct_b": other["Unique_ID"],
                "shared_boundary_ft": shared_length
            })
        


In [44]:
print(f"total neighbors: {len(ar_neighbors)}")

total neighbors: 7260


In [45]:
edges = pd.DataFrame(ar_neighbors)
edges.head()

,precinct_a,precinct_b,shared_boundary_ft
0,05001-81 - Stuttgart 2,05001-41 - Morris,25488.828320
1,05001-81 - Stuttgart 2,05001-91 - Stuttgart 3,12281.342283
2,05001-81 - Stuttgart 2,05001-71 - Stuttgart 1,6494.174373
3,05001-81 - Stuttgart 2,05001-42 - Gum Pond,12601.386044
4,05001-36 - Gillett Ward 3,05001-35 - Gillett Ward 2,6071.462424


In [50]:
a_cnt = edges['precinct_a'].value_counts()
b_cnt = edges['precinct_b'].value_counts()
neighbor_counts = a_cnt.add(b_cnt, fill_value=0)
neighbor_counts = neighbor_counts.reset_index()
neighbor_counts.columns = ["Unique_ID", "num_neighbors"]

In [51]:
neighbor_counts

,Unique_ID,num_neighbors
0,05001-11 - McFall,5.0
1,05001-12 - Crockett,3.0
2,05001-13 - Keaton,8.0
3,05001-14 - Mill Bayou,7.0
4,05001-15 - Almyra,1.0
...,...,...
2676,05149-24 - Waveland,5.0
2677,05149-25 - Riley,5.0
2678,05149-26 - Rover,5.0
2679,05149-28 - Ward,5.0


In [52]:
ar_df.shape

(2681, 3)

In [55]:
neighbor_counts.shape

(2681, 2)

In [57]:
neighbor_counts.to_csv("output/Arkansas/ar_precinct_neighbors_cnt.csv", index=False)
edges.to_csv("output/Arkansas/ar_precinct_neighbors_map.csv", index=False)